In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100  # MB
from IPython.display import display
%matplotlib inline
%matplotlib notebook
from IPython.display import display, HTML
%matplotlib agg
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, display
from tensorflow.keras.datasets import mnist
from sklearn.decomposition import PCA
# =====================================
# 1. Load MNIST
# =====================================
(train_images, train_labels), _ = mnist.load_data()
# =====================================
# 2. Preprocess (Binary + PCA)
# =====================================
# Flatten
X = train_images.reshape(len(train_images), -1)
# Normalize
X = X / 255.0
# Select digits 0 and 1
mask = (train_labels == 0) | (train_labels == 1)
X = X[mask]
y = train_labels[mask]
# Convert to 0/1
y = (y == 1).astype(int)
# PCA → 2D
pca = PCA(n_components=2)
X = pca.fit_transform(X)
N = len(X)
# =====================================
# 3. Logistic Regression
# =====================================
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
def predict(w1, w2, X):
    z = w1*X[:,0] + w2*X[:,1]
    return sigmoid(z)
def loss(w1, w2):
    y_hat = predict(w1, w2, X)
    eps = 1e-8
    return -np.mean(
        y*np.log(y_hat+eps) +
        (1-y)*np.log(1-y_hat+eps)
    )
def gradients(w1, w2):
    y_hat = predict(w1, w2, X)
    err = y_hat - y
    g1 = np.mean(err * X[:,0])
    g2 = np.mean(err * X[:,1])
    return g1, g2
# =====================================
# 4. Hyperparameters
# =====================================
lr = 0.1
iters = 200
momentum = 0.9
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
w1_init = 0.0
w2_init = 0.0
# =====================================
# 5. Optimizers
# =====================================
def GD():
    w1, w2 = w1_init, w2_init
    path, losses = [], []
    for _ in range(iters):
        g1, g2 = gradients(w1, w2)
        w1 -= lr*g1
        w2 -= lr*g2
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def SGD():
    w1, w2 = w1_init, w2_init
    path, losses = [], []
    for _ in range(iters):
        i = np.random.randint(N)
        xi = X[i]
        yi = y[i]
        z = w1*xi[0] + w2*xi[1]
        yh = sigmoid(z)
        err = yh - yi
        w1 -= lr*err*xi[0]
        w2 -= lr*err*xi[1]
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def MiniBatch(batch=64):
    w1, w2 = w1_init, w2_init
    path, losses = [], []
    for _ in range(iters):
        idx = np.random.choice(N, batch)
        xb = X[idx]
        yb = y[idx]
        z = w1*xb[:,0] + w2*xb[:,1]
        yh = sigmoid(z)
        err = yh - yb
        g1 = np.mean(err * xb[:,0])
        g2 = np.mean(err * xb[:,1])
        w1 -= lr*g1
        w2 -= lr*g2
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def Nesterov():
    w1, w2 = w1_init, w2_init
    v1 = v2 = 0
    path, losses = [], []
    for _ in range(iters):
        lw1 = w1 - momentum*v1
        lw2 = w2 - momentum*v2
        g1, g2 = gradients(lw1, lw2)
        v1 = momentum*v1 + lr*g1
        v2 = momentum*v2 + lr*g2
        w1 -= v1
        w2 -= v2
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def Adagrad():
    w1, w2 = w1_init, w2_init
    h1 = h2 = 0
    path, losses = [], []
    for _ in range(iters):
        g1, g2 = gradients(w1, w2)
        h1 += g1**2
        h2 += g2**2
        w1 -= lr*g1/(np.sqrt(h1)+eps)
        w2 -= lr*g2/(np.sqrt(h2)+eps)
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def RMSProp():
    w1, w2 = w1_init, w2_init
    h1 = h2 = 0
    rho = 0.9
    path, losses = [], []
    for _ in range(iters):
        g1, g2 = gradients(w1, w2)
        h1 = rho*h1 + (1-rho)*g1**2
        h2 = rho*h2 + (1-rho)*g2**2
        w1 -= lr*g1/(np.sqrt(h1)+eps)
        w2 -= lr*g2/(np.sqrt(h2)+eps)
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def Adam():
    w1, w2 = w1_init, w2_init
    m1 = m2 = v1 = v2 = 0
    path, losses = [], []
    for t in range(1, iters+1):
        g1, g2 = gradients(w1, w2)
        m1 = beta1*m1 + (1-beta1)*g1
        m2 = beta1*m2 + (1-beta1)*g2
        v1 = beta2*v1 + (1-beta2)*g1**2
        v2 = beta2*v2 + (1-beta2)*g2**2
        m1h = m1/(1-beta1**t)
        m2h = m2/(1-beta1**t)
        v1h = v1/(1-beta2**t)
        v2h = v2/(1-beta2**t)
        w1 -= lr*m1h/(np.sqrt(v1h)+eps)
        w2 -= lr*m2h/(np.sqrt(v2h)+eps)
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
# =====================================
# 6. Loss Surface
# =====================================
w1s = np.linspace(-5, 5, 150)
w2s = np.linspace(-5, 5, 150)
W1, W2 = np.meshgrid(w1s, w2s)
Z = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]): 
        Z[i,j] = loss(W1[i,j], W2[i,j])
# =====================================
# 7. Animations
# =====================================
def animate_2D_inline(name, path):
    fig, ax = plt.subplots()
    ax.contour(W1, W2, Z, 40)
    ax.set_title(f"{name} - 2D Contour")
    line, = ax.plot([], [], 'r-')
    point, = ax.plot([], [], 'ro')
    def update(i):
        xs = [p[0] for p in path[:i]]
        ys = [p[1] for p in path[:i]]
        if len(xs)==0:
            return line, point
        line.set_data(xs, ys)
        point.set_data([xs[-1]], [ys[-1]])
        return line, point
    anim = animation.FuncAnimation(
        fig, update,
        frames=range(1,len(path)+1),
        interval=60,
        blit=True
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())
def animate_3D_inline(name, path):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(W1, W2, Z, cmap="viridis", alpha=0.7)
    ax.set_title(f"{name} - 3D Surface")
    point, = ax.plot([], [], [], 'ro')
    def update(i):
        w1, w2 = path[i-1]
        z = loss(w1, w2)
        point.set_data([w1],[w2])
        point.set_3d_properties([z])
        return point,
    anim = animation.FuncAnimation(
        fig, update,
        frames=range(1,len(path)+1),
        interval=60,
        blit=True
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())
def plot_loss_inline(name, losses):
    fig, ax = plt.subplots()
    ax.plot(losses, lw=2)
    ax.set_title(f"{name} - Loss Curve")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.grid(True)
    plt.close(fig)   # Prevent duplicate rendering
    display(fig)
path, l = GD()
display(animate_2D_inline("GD", path))
display(animate_3D_inline("GD", path))
plot_loss_inline("GD", l)

path, l = SGD()
display(animate_2D_inline("SGD", path))
display(animate_3D_inline("SGD", path))
plot_loss_inline("SGD", l)

path, l = MiniBatch()
display(animate_2D_inline("MiniBatch", path))
display(animate_3D_inline("MiniBatch", path))
plot_loss_inline("MiniBatch", l)

path, l = Nesterov()
display(animate_2D_inline("Nesterov", path))
display(animate_3D_inline("Nesterov", path))
plot_loss_inline("Nesterov", l)

path, l = Adagrad()
display(animate_2D_inline("Adagrad", path))
display(animate_3D_inline("Adagrad", path))
plot_loss_inline("Adagrad", l)

path, l = RMSProp()
display(animate_2D_inline("RMSProp", path))
display(animate_3D_inline("RMSProp", path))
plot_loss_inline("RMSProp", l)

path, l = Adam()
display(animate_2D_inline("Adam", path))
display(animate_3D_inline("Adam", path))
plot_loss_inline("Adam", l) 